# 04 · What trading actually costs

Every trade pays three things on top of the price move:

| Cost | What it is | How we get it |
|---|---|---|
| Commission | the broker's fee per order | broker's price list |
| Bid-ask spread | buying at the ask and selling at the bid, which is lower | **estimated** from daily prices |
| Transaction tax | 0.2% on purchases of large Spanish companies | the tax agency's yearly lists |

**Assumed account:** €100,000 split into 10 slots of €10,000, one per stock. That sets the **trade size: €10,000**, which matters because commissions have a minimum fee.

The code is in [`src/costs.py`](../src/costs.py). This notebook explains each piece and applies it to the in-sample trades only.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from bidask import edge

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
from costs import TRADE_SIZE, add_spread, commission_rate, net_returns
from vwap_dip import SPLIT, START, run

SYMBOLS = pd.read_csv(ROOT / "data" / "universe" / "universe_v1.csv")["yahoo_symbol"].tolist()
bars, all_trades = run(SYMBOLS)
trades = all_trades[(all_trades["entry_date"] >= START) & (all_trades["entry_date"] < SPLIT)].reset_index(drop=True)
print(len(trades), "in-sample trades")

225 in-sample trades


## Part A · Commission

Interactive Brokers' fixed pricing for Spanish stocks is **0.05% of the trade value, with a €3 minimum** per order ([Rankia's review](https://www.rankia.com/blog/mejores-brokers/4652044-analisis-interactive-brokers-regulacion-mercados-comisiones-mucho-mas?page=2); IBKR's own page blocks automated access, so check it directly before the write-up).

The minimum makes small trades expensive. Below €6,000 the €3 floor is more than 0.05%:

In [2]:
pd.Series({f"€{size:,}": 100 * commission_rate(size) for size in [1_000, 3_000, 6_000, 10_000, 50_000]},
          name="commission per order, % of trade").round(3)

€1,000     0.30
€3,000     0.10
€6,000     0.05
€10,000    0.05
€50,000    0.05
Name: commission per order, % of trade, dtype: float64

At our €10,000 trade size, that's €5 per order: **0.05% to buy and 0.05% to sell**.

## Part B · Transaction tax

The tax applies to purchases of a stock in any year it's on the Agencia Tributaria's list, which is based on market value on 1 December of the year before. The lists for every year since the tax began were downloaded and searched for our 10 companies:

| Tax year | List based on market value on | Our stocks on it |
|---|---|---|
| 2021 | 16 Dec 2020 | none |
| 2022 | 1 Dec 2021 | none |
| 2023 | 1 Dec 2022 | none |
| 2024 | 1 Dec 2023 | none |
| 2025 | 1 Dec 2024 | none |
| 2026 | 1 Dec 2025 | none |

So the tax is zero for every trade. The code still has the hook (`TAXED` in `costs.py`), so a future universe that includes a taxed stock is charged correctly. Sources are in [`data/reference/SOURCES.md`](../data/reference/SOURCES.md).

## Part C · The bid-ask spread

At any moment there's a best price someone will sell at (the **ask**) and a best price someone will buy at (the **bid**). The ask is higher. Buy with a market order and you pay the ask; sell straight away and you get the bid. The gap is the **spread**, and a round trip loses all of it. Each side costs **half the spread** relative to the midpoint.

Daily bars don't record the bid or the ask. But the spread leaves a fingerprint: trades bounce between bid and ask, so the high is pushed up, the low down, and the close lands on one side or the other at random. Researchers have turned that fingerprint into formulas. The simplest is **Abdi & Ranaldo (2017)**:

$$\text{spread}^2 = 4 \cdot \text{average}\big[(c_t - \eta_t)(c_t - \eta_{t+1})\big]$$

where $c_t$ is the log close and $\eta_t$ is the midpoint of the log high and log low. The true price is unknown, so the midpoint of the high and low stands in for it. The close sits half a spread above or below the true price, and averaging the product picks out that half-spread squared.

**How can we tell whether a formula like this works?** Build a fake market where we *know* the spread, generate daily bars from it, and see whether the formula gets the answer back.

In [3]:
rng = np.random.default_rng(0)


def fake_market(spread, days=252, trades_per_day=300, daily_vol=0.025):
    """Daily bars from a made-up market with a known spread: every trade happens at the bid or the ask."""
    rows, mid = [], 0.0
    for _ in range(days):
        path = mid + np.cumsum(rng.normal(0, daily_vol / np.sqrt(trades_per_day), trades_per_day))  # log mid-price
        prices = path + rng.choice([-1, 1], trades_per_day) * spread / 2  # each trade half a spread above or below
        rows.append((prices[0], prices.max(), prices.min(), prices[-1]))
        mid = path[-1]
    return pd.DataFrame(rows, columns=["open", "high", "low", "close"]).apply(np.exp)


def abdi_ranaldo(b):
    c = np.log(b["close"])
    mid_range = (np.log(b["high"]) + np.log(b["low"])) / 2
    s2 = 4 * ((c - mid_range) * (c - mid_range.shift(-1))).mean()
    return np.sqrt(max(s2, 0))  # noise can make the average negative; that reads as a spread of 0


rows = {}
for true_spread in [0.003, 0.006, 0.010]:
    markets = [fake_market(true_spread) for _ in range(100)]  # 100 fake years
    for name, est in [("Abdi-Ranaldo", [abdi_ranaldo(b) for b in markets]),
                      ("EDGE", [edge(b["open"], b["high"], b["low"], b["close"]) for b in markets])]:
        est = 100 * np.array(est)
        rows[(f"{100 * true_spread:.1f}%", name)] = {"average estimate %": est.mean(),
                                                     "10th pct %": np.percentile(est, 10),
                                                     "90th pct %": np.percentile(est, 90),
                                                     "years estimated as 0": (est == 0).mean()}
check = pd.DataFrame(rows).T
check.index.names = ["true spread", "estimator"]
check.round(2)

average estimate %  10th pct %  90th pct %  \
true spread estimator                                                  
0.3%        Abdi-Ranaldo                0.31        0.00        0.79   
            EDGE                        0.31        0.18        0.43   
0.6%        Abdi-Ranaldo                0.59        0.00        1.10   
            EDGE                        0.58        0.43        0.70   
1.0%        Abdi-Ranaldo                0.93        0.50        1.30   
            EDGE                        1.00        0.88        1.11   

                          years estimated as 0  
true spread estimator                           
0.3%        Abdi-Ranaldo                  0.45  
            EDGE                          0.00  
0.6%        Abdi-Ranaldo                  0.24  
            EDGE                          0.00  
1.0%        Abdi-Ranaldo                  0.06  
            EDGE                          0.00

Both estimators are right **on average**. But Abdi-Ranaldo is very noisy over a single year: for small spreads, a large share of fake years come out as 0. **EDGE** (Ardia, Guidotti & Kroencke, *Journal of Financial Economics*, 2024) also uses the open and is far more precise, with the 10th–90th percentile range within about ±0.1–0.2 points. We use EDGE, through the authors' own `bidask` package.

Each trade's spread comes from the **previous 252 trading days** only (`add_spread` in `costs.py`).

In [4]:
bars = {s: add_spread(b) for s, b in bars.items()}

spread_by_year = pd.DataFrame({s: 100 * b["spread"].loc[START:SPLIT].groupby(b.loc[START:SPLIT].index.year).median()
                               for s, b in bars.items()}).T
spread_by_year["median"] = pd.Series({s: 100 * b["spread"].loc[START:SPLIT].median() for s, b in bars.items()})
spread_by_year.round(2)

Date,2016,2017,2018,2019,2020,2021,2022,2023,median
OHLA.MC,0.44,0.72,1.27,1.89,0.76,1.17,1.22,0.81,1.11
EDR.MC,1.74,1.58,1.30,1.96,1.51,2.01,1.66,1.97,1.70
AMP.MC,1.46,0.94,0.71,0.42,0.92,1.27,1.76,1.37,1.05
ORY.MC,0.52,0.96,1.37,0.79,0.99,1.15,1.25,0.78,1.05
TUB.MC,1.56,1.25,0.91,1.55,1.99,1.60,1.71,1.49,1.54
ENC.MC,1.42,0.90,0.68,0.34,0.75,0.83,1.02,0.78,0.78
NXT.MC,2.92,1.91,2.14,2.47,2.70,2.59,2.75,2.14,2.42
DOM.MC,1.82,1.61,1.31,1.29,2.20,1.76,1.62,1.80,1.63
GSJ.MC,1.29,0.62,1.01,1.22,1.34,1.81,1.64,1.56,1.35
ADX.MC,1.60,1.56,2.33,0.71,1.41,0.94,1.30,1.20,1.40


Estimated spreads of roughly **0.8% to 2.4%**. These are small, low-priced stocks, where one price step (tick) can itself be a large share of the price. Treat these as estimates, not facts.

### Do we even pay the spread?

Our rules buy and sell **at the open**. On BME, the open is set by an **opening auction**: orders collect before 9:00 and all of them trade at one price. An order in the auction gets that price, with no bid and ask to cross. So there are two honest scenarios:

| Scenario | We pay | Fair when |
|---|---|---|
| **Auction fills** | commission only | our €10,000 order is too small to move the auction price |
| **Cross the spread** | commission + half the spread on each side | we trade with market orders after the open, or our order does move the auction |

The truth is probably in between. The less a stock trades, the more a €10,000 order matters, and for the quieter names that's around 3% of a typical day's trading (notebook 01).

## Part D · In-sample results after costs

In [5]:
trades["net_auction"] = net_returns(trades, bars, cross_spread=False)
trades["net_spread"] = net_returns(trades, bars, cross_spread=True)


def describe(r):
    return {"EV per trade %": 100 * r.mean(), "median %": 100 * r.median(), "win rate %": 100 * (r > 0).mean()}


pd.DataFrame({
    "before costs": describe(trades["return"]),
    "auction fills": describe(trades["net_auction"]),
    "cross the spread": describe(trades["net_spread"]),
}).T.round(2)

,EV per trade %,median %,win rate %
before costs,-0.29,0.27,52.00
auction fills,-0.39,0.17,51.56
cross the spread,-1.73,-1.19,42.67


In [6]:
cost = pd.DataFrame({
    "auction fills": 100 * (trades["return"] - trades["net_auction"]),
    "cross the spread": 100 * (trades["return"] - trades["net_spread"]),
})
cost.describe().loc[["mean", "min", "50%", "max"]].rename(index={"50%": "median"}).round(2).rename_axis("round-trip cost %")

,auction fills,cross the spread
round-trip cost %,,
mean,0.10,1.44
min,0.06,0.26
median,0.10,1.40
max,0.14,3.24


## Takeaways

1. **Commission is small** at this trade size: 0.10% for a round trip. It would matter more for smaller trades because of the €3 minimum.
2. **The tax is zero** for all 10 stocks in every year. Choosing stocks below the €1bn line did its job.
3. **The spread is the big unknown.** Depending on how orders fill, a round trip costs from 0.10% to about 1.4% on average. That's more than the size of any realistic edge for a strategy like this.
4. **Version 1 loses money in-sample under both scenarios.** Next, notebook 05 runs the frozen rules and cost model once on the out-of-sample years.